[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Data-Crew/transport-networks-lab/blob/main/Notebooks/Practica_guiada3_IntroOSMNx194.ipynb)


#  Grafos y redes de transporte

##  OSMnx Parte I
### Calles y límites: trabajando el callejero como una red de nodos y arcos.

#### Algunos conceptos básicos

Si recordamos nuestras clases anteriores, llamamos grafo a un tipo de estructura de datos compuesta por un conjunto de nodos o vértices unidos a través de ejes o arcos. Algo que resulta particularmente práctico para diagramar redes de distinto tipo.

[OSMNx](https://osmnx.readthedocs.io/en/stable/index.html) es una librería de python creada por [gboeing](https://geoffboeing.com/about/). La misma permite construir redes de calles a partir de la API de [OpenStreeMap](https://www.openstreetmap.org). Utilizando, ya sea geometrías provistas como argumentos o bien coordenadas específicas, esta permite tanto la reproyección como la visualización del objeto resultante.

Para ello, trabaja básicamente a partir de la librería que vimos en encuentros anteriores: [NetworkX](https://networkx.github.io/documentation/stable/)

In [ ]:
#%%capture
!pip install osmnx==1.9.4

En la propuesta que preparamos para esta clase, vamos enfocarnos en dos de las principales funcionalidades de `OMSNx`. En primeria instancia, vamos a revisar algunos de sus métodos para definir un escenario territorial específico. Es decir, vamos a ver cómo trabajar con geometrias. Una vez repasado esto, nos vamos a adentrar en cuestiones específicas del análisis y descripción de una red de calles.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import osmnx as ox
print(ox.__version__)

Algo interesante de `OSMnx` es que se pueden ajustar algunas configuraciones relacionadas a su uso (ver [config](https://osmnx.readthedocs.io/en/stable/osmnx.html#osmnx.utils.config) y [settings](https://osmnx.readthedocs.io/en/stable/user-reference.html#module-osmnx.settings))

Nosotros vamos a definir los parametros `log_file` y `use_cache` como `True` porque:


* log_file (bool) – if True, save log output to a file in logs_folder

* use_cache (bool) – if True, cache HTTP responses locally instead of calling API repeatedly for the same request


Si, por ejemplo, hubiesemos setado `use_cache` como `False` OSMnx no guardaria las respuestas del server para evitar llamarlo repetidamente en la misma query cada vez que se ejecuta algún método. Este tipo de configuraciones se pueden ajustar manualmente. Cualquier parámetro no especificado tomara el valor por default de la configuración global).

In [ ]:
ox.config(log_file=True, use_cache=True)

In [ ]:
ox.settings.log_file=True
ox.settings.use_cache=True

Actualmente, `OSMNx` se encuentra en un proceso de migración hacia su versión `2.0`. Muchas de las funcionalidades que revisaremos en este notebook no se encuentran totalmente migradas. A l@s que les interese seguir trabajando con esta librería les sugerimos que estén al corriente de [este issue](https://github.com/gboeing/osmnx/issues/1123).

Para evitar los `Future` y `Deprecation` warnings vamos a desactivarlos por el momento.

In [ ]:
#import sys
#import io

# Redireccionar los warnings
#sys.stderr = io.StringIO()

## Sección 1: Descargando geometrías

### 1.1. Entidades territoriales

Antes de comenzar a trabajar con calles, debemos saber que OSMNx nos permite descargar distintos tipos de entidades espaciales con el módulo `features`. A través del mismo, se pueden consultar desde paradas de tránsito a distintos puntos de intéres como escuelas, negocios, edificios, barrios, etc.

Para hacer una consulta a la API, debemos utilizar un diccionario con tags de OSM. Este debe especificar un string con el nombre del tag como key.

Respecto de los values, estos pueden ser un booleano para devolver todos los objetos que en Openstreet usan dicho tag. También puede ser otro string con el nombre de una subcategoría con la que matchea el tag que seteamos en la key, o una lista de strings para devolver al menos alguna de las subcategorías con las que queremos que matchee nuestro tag.

[MapFeatures de OSM](https://wiki.openstreetmap.org/wiki/Map_features)

In [ ]:
# Digamos que queremos consultar entidades que hayan sido taggeadas como edificios

from shapely.geometry import box

def get_features_safe(place, tags=None, buffer_km=1.0):
    """
    Obtiene features o geometría de un lugar incluso si no tiene polígono definido.
    Si tags=None, devuelve el polígono (real o bounding box).
    """
    try:
        if tags:
            return ox.features_from_place(place, tags)
        else:
            return ox.geocode_to_gdf(place)
    except TypeError:
        print(f"⚠️ '{place}' no tiene geometría poligonal; usando bounding box aproximado.")
        gdf_point = ox.geocode_to_gdf(place, which_result=1)
        lon, lat = gdf_point.geometry.iloc[0].x, gdf_point.geometry.iloc[0].y
        delta = buffer_km / 111
        bbox = box(lon - delta, lat - delta, lon + delta, lat + delta)
        return ox.features_from_polygon(bbox, tags) if tags else gdf_point.set_geometry([bbox])

# Ejemplo: edificios en José León Suárez
localidad = "José León Suárez, Partido de General San Martín, Buenos Aires, Argentina"
tags = {"building": True}

edificios = get_features_safe(localidad, tags, buffer_km=1.5)
print(f"{edificios.shape[0]} entidades descargadas.")
edificios.head()

In [ ]:
type(edificios)

In [ ]:
import pandas as pd

# 1️⃣ Buscar polígonos de uso residencial o basural
tags_landuse = {'landuse': ['residential', 'landfill']}
usos_suelo = get_features_safe(localidad, tags_landuse, buffer_km=2.0)

# 2️⃣ Buscar entidades tipo 'barrio' o 'vecindario'
tags_place = {'place': 'neighbourhood'}
barrios = get_features_safe(localidad, tags_place, buffer_km=2.0)

# 3️⃣ Unir ambas capas
combined = (
    pd.concat([usos_suelo, barrios])
    .reset_index(drop=True)
    .drop_duplicates(subset=['geometry'])
)

# 4️⃣ Filtrar por nombre (incluso parciales)
villa_hidalgo = combined[combined['name'].str.contains('Hidalgo', case=False, na=False)]
print(f"{villa_hidalgo.shape[0]} entidades encontradas con 'Hidalgo'")
villa_hidalgo[['name','landuse','place','geometry']].head()

Veamos en qué consisten las geometrías que hemos descargado desde OSM!

In [ ]:
f, ax = plt.subplots(figsize=(12, 7))

# --- 1️⃣ Definir subconjuntos ---
usos_residenciales = usos_suelo.loc[usos_suelo.landuse == "residential"]
basurales = usos_suelo.loc[usos_suelo.landuse == "landfill"]

# En lugar de vh_polyg (vacío), usamos los polígonos con “Hidalgo” en el nombre
vh_polyg = combined[
    combined["name"].str.contains("Hidalgo", case=False, na=False)
    & (combined.geometry.type.isin(["Polygon", "MultiPolygon"]))
]

# --- 2️⃣ Ploteo ordenado (de fondo a frente) ---
if not usos_residenciales.empty:
    usos_residenciales.plot(ax=ax, color="lightgrey", edgecolor="none", label="Residencial")

if not basurales.empty:
    basurales.plot(ax=ax, color="saddlebrown", edgecolor="none", label="Basural")

if not edificios.empty:
    edificios.plot(ax=ax, color="black", markersize=1, alpha=0.5, label="Edificios")

if not vh_polyg.empty:
    vh_polyg.boundary.plot(ax=ax, color="red", linewidth=2.5, label="Villa Hidalgo Sur")

ax.set_axis_off()
ax.legend(loc="upper right")
plt.title("Villa Hidalgo Sur y usos de suelo en José León Suárez", fontsize=14)
plt.show()

### 1.2. Límites territoriales desde OpenStreetMap

In [ ]:
localidades = [
    "José León Suárez, Buenos Aires, Argentina",
    "Loma Hermosa, Buenos Aires, Argentina",
    "Villa Ballester, Buenos Aires, Argentina",
]

tags = {"boundary": "administrative"}

resultados = []
for loc in localidades:
    try:
        gdf = ox.features_from_place(loc, tags)
        if not gdf.empty:
            gdf["localidad"] = loc
            resultados.append(gdf)
            print(f"✅ {loc}: {len(gdf)} límite(s) encontrado(s)")
        else:
            print(f"⚠️ {loc}: sin límites administrativos en OSM.")
    except Exception as e:
        print(f"❌ {loc}: error -> {e}")

# Unimos los resultados
if resultados:
    sm_norte = gpd.GeoDataFrame(pd.concat(resultados, ignore_index=True))
else:
    sm_norte = gpd.GeoDataFrame()

In [ ]:
# Mostrar en mapa si se encontró algo
if not sm_norte.empty:
    fig, ax = plt.subplots(figsize=(8, 6))

    # Plot principal
    sm_norte.plot(
        column="name",
        cmap="tab20b",       # paleta amplia (20 colores)
        alpha=0.6,
        edgecolor="black",
        linewidth=0.7,
        legend=True,
        ax=ax
    )

    # Título y formato
    ax.set_title("Límites administrativos encontrados en OSM", fontsize=16, pad=7)
    ax.axis("off")

    # 💡 Mover la leyenda fuera del mapa
    leg = ax.get_legend()
    if leg:
        leg.set_bbox_to_anchor((2., 0.75))  # mover a la derecha
        leg.set_frame_on(True)
        leg.get_frame().set_alpha(0.9)
        leg.set_title("Localidades")

        # ✅ forma compatible de cambiar tamaño del título
        leg.get_title().set_fontsize(11)

        # Reducir tamaño de los nombres
        for text in leg.get_texts():
            text.set_fontsize(9)

    plt.tight_layout()
    plt.show()
else:
    print("Ninguna de las localidades tiene límites administrativos cargados en OSM.")

In [ ]:
sm_norte

## Sección 2: Construyendo redes de calles

OSMnx también cuenta con un módulo específico de grafos - [acá el hipervínculo para revisar todos sus métodos](https://osmnx.readthedocs.io/en/stable/osmnx.html#module-osmnx.graph) - que permite descargar una red de calles desde la API de OSM y modelarla a partir de diferentes tipologías. Los nodos se establecen a partir de la intersección de calles, mientras que los ejes o arcos que los unen son las calles en sí mismas. Existen diversas maneras de descargar una red de calles, ya sea desde un conjunto de coordenadas, un polígono específico o directamente el nombre de una unidad administrativa registrada. A lo largo del notebook iremos explorando cada una de ellas.

A su vez, cada uno de estos métodos nos brindarán la posibilidad de definir diferentes tipos de redes:

* 'drive' - calles o vias de tránsito públicas (sin contar colectoras o caminos paralelos)
* 'drive_service' - calles de tránsito, incluyendo colectoras )o service roads=
* 'walk' - calles y caminos también disponibles para peatones (este tipo de red ignora la direccionalidad)
* 'bike' - calles y caminos con bicisendas
* 'all' - descarga cualquier tipo de calle o camino público
* 'all_private' - descarga cualquier tipo de calle o camino, inluyendo los de acceso privado.

### 2.1. Calles a partir de coordenadas específicas

El método `graph_from_bbox` permite consultar todos los nodos y calles de la API de OSM dentro de un recuadro delimitado por un conjunto de coordenadas arbitrarias. Existen varias herramientas para consultar coordenadas de algún lugar que nos interese, [bbox finder](http://bboxfinder.com/) es una de ellas.

Ahora vamos a consultar las coordenadas del polígono que contiene algunos asentamientos informales del partido de San Martín, en la provincia de Buenos Aires. [Delimitemos Villa Hidalgo y La Cárcova](http://bboxfinder.com/#-34.533783,-58.589787,-34.506274,-58.566399)

In [ ]:
# establecemos un conjunto de coordenadas arbitrarias
norte, sur, este, oeste = -34.506274, -34.533818, -58.566442, -58.589830

# instanciamos nuestro grafo con bounding box
G = ox.graph_from_bbox(norte, sur, este, oeste, network_type='drive_service')

In [ ]:
# Vemos que es un grafo dirigido!
type(G)

In [ ]:
fig, ax = ox.plot_graph(G, bgcolor='w',node_color='r')

Otra alternativa para consultar redes de calles a partir de conjuntos de coordenadas es utilizar localizaciones específcias. Es decir, en lugar de definir un polígono determinado, utilizar un punto de referencia. Esto se consigue con el método `graph_from_point` y lo que hace es crear un *bounding box* o marco de referencia *n* metros hacia el norte, sur, este y oeste según lo determinemos en el parámetro `dist`.

In [ ]:
# creamos una tupla en el centroide de Villa Hidalgo
referencia = (-34.50944,-58.58610)

# a partir del punto de referencia previo, se construye una red dentro de un marco de 500m hacia el N,S,E y O.
G2 = ox.graph_from_point(center_point=referencia, dist=500, dist_type='bbox', network_type='drive')

In [ ]:
ox.graph_from_point?

In [ ]:
fig2, ax2 = ox.plot_graph(G2, bgcolor='white', node_color='red', node_size=35)

Si prestan atención, verán que utilizamos el parámetro `dist_type`. Además del valor *bbox* este nos ofrece la opción *network* ...

In [ ]:
G3 = ox.graph_from_point(center_point=referencia, dist=500, dist_type='network')
fig3, ax3 = ox.plot_graph(G3, bgcolor='white', node_color='red', node_size=35)

En el ejemplo anterior hicimos dos modificaciones. Primero, cambiamos el tipo de distancia. Segundo, utilizamos el default de `network_type`.

Aclaremos primero esto último. Cuando utilizamos *all_private* (que es el valor por defecto) para definir el tipo de red, no estamos filtrando o exluyendo vías que restringen la circulación para determinado tipo de tráfico. Pueden ver que en el último plot, se agregan algunas calles de traza irregular. Es decir, ampliamos las posibilidades de circulación.

La diferencia respecto de utilizar *bbox*, es que *network* remueve los nodos más allá de *n* metros de distancia a lo largo de la red desde el punto que tomamos como referencia. Es decir, en ambas opciones se establece un marco espacial hacia el norte, sur, este y oeste del mismo y luego se construye la red con todos los nodos y vías disponibles en la API de OSM dentro del marco establecido. Pero si utilizamos *network*, los nodos más allá de cierta distancia de viaje (o distancia dentro de la red) serán removidos.

También es importante remarcar que al ser *network* el tipo de distancia se respeta el sentido de las calles. En el caso de nuestro ejemplo, el radio de 500 metros de distancia que establecimos toma en cuenta los nodos a los que se puede llegar desde el punto de referencia viajando en el sentido permitido y no en el contrario.

In [ ]:
# creamos una nueva red de calles, esta vez restringiendola a vias caminables
G4 = ox.graph_from_point(center_point=referencia, dist=500, dist_type='network', network_type='walk')
fig4, ax4 = ox.plot_graph(G4, bgcolor='white', node_color='red', node_size=35)

Al definir el tipo de red como *walk*, hacemos que los ejes sean solamente calles donde está permitido caminar. En otras palabras, construimos un grafo dirigido con vínculos bidireccionales entre nodos. Básicamente, porque se puede llegar a cualquier intersección en ambas direcciones del sentido único de la calle. Por lo tanto, los 500 m ahora tienen en cuenta los nodos a los que puede llegar mientras viaja en cualquier dirección (incluso si es una calle de un solo sentido).

### 2.2. Calles a partir de nombres y geometrías

Además del uso de coordenadas, OSMNx permite consultar calles que responden a nombres específicos o que se encuentran dentro de los límites de polígonos que especifiquemos.

En el primer caso, al pasar el nombre de una calle, esta se geocodifica y se crea un marco espacial o bounding box. Luego se descarga la red lista para trabajar como vimos en los ejemplos anteriores. También se puede especificar el nombre de la localidad, partido o unidad administrativa cuya red de calles nos resulte de interés.

In [ ]:
# definimos una direccion de referencia
direccion = 'Doctor Ricardo Balbin, Partido de General San Martín, Buenos Aires, Argentina'

In [ ]:
G5 = ox.graph_from_address(address= direccion,
                           dist=1000, dist_type='network', network_type='drive')

# así como vimos con la descarga de geometrías, los grafos también pueden ser reproyectados en UTM
G5_prj = ox.project_graph(G5)

In [ ]:
fig5, ax5 = ox.plot_graph(G5_prj,  bgcolor='white', node_color='blue', node_size=35)

En este caso, lo que estamos haciendo es crear una red a partir de una calle y altura específicas. Eso se toma como punto de referencia y se consideran solamente los nodos que están a 1km de distancia desde esa dirección a lo largo de la red.

Como mencionamos, además del método `graph_from_adress`, también se puede utilizar `graph_from_place`. Esto con el objetivo de indicar el nombre del lugar en el que estemos interesados. El cual puede ser ...

In [ ]:
# algún barrio o asentamiento
G6 = ox.graph_from_place('Villa Hidalgo, José León Suárez, Partido de General San Martín, Buenos Aires',
                         network_type='walk')

In [ ]:
fig6, ax6 = ox.plot_graph(G6, bgcolor='white', node_color='blue', node_size=35)

In [ ]:
G7 = ox.graph_from_place('Villa La Carcova, Partido de General San Martín, Buenos Aires',
                         network_type='walk')

In [ ]:
fig7, ax7 = ox.plot_graph(G7, bgcolor='white', node_color='red', node_size=35)

Por último, vamos a mencionar el método `graph_from_polygon`. Vimos que podemos construimos grafos a partir de nombres de calles o lugares específicos. Este método nos permite hacer lo mismo, pero con la diferencia de ser nosotros los que definimos a partir de una geometría los límites de descarga de nuestra red.

In [ ]:
from google.colab import drive
drive.mount('/drive/')

In [ ]:
path = '/drive/MyDrive/Técnicas y Análisis de datos del Transporte/Data/renabap.geojson'
barrios = gpd.read_file(path)

# nos quedamos con los barrios de San Martin
barrios_sm = barrios.loc[(barrios.provincia == 'Buenos Aires') & (barrios.departamen == 'General San Martín')]

In [ ]:
# y ahora con algunos de jose leon suarez
barrios_ls = barrios_sm.loc[(barrios_sm.nombre_bar == 'Villa Hidalgo')|
                            (barrios_sm.nombre_bar == 'La Carcova')].copy()

In [ ]:
f, ax = plt.subplots()
barrios_ls.plot(ax=ax)
ax.set_axis_off();

In [ ]:
# Villa Hidalgo
vh = barrios_ls['geometry'].iloc[0]
G9 = ox.graph_from_polygon(vh, network_type='walk')

fig9, ax9 = ox.plot_graph(G9, bgcolor='white', node_color='blue', node_size=35)

In [ ]:
# La Carcova
lc = barrios_ls['geometry'].iloc[1]
G10 = ox.graph_from_polygon(lc, network_type='walk')

fig10, ax10 = ox.plot_graph(G10, bgcolor='white', node_color='red', node_size=35)